In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
#import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from keras.optimizers import Adam
from keras.losses import Loss
from keras.initializers import GlorotUniform, GlorotNormal
#from keras.utils import to_categorical
from joblib import Parallel, delayed, parallel_backend
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

## GPU visibility

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
#print(tf.config.experimental.list_physical_devices())
#gpus = tf.config.list_physical_devices('GPU'); print(gpus)
#tf.config.set_visible_devices([gpus[1]], 'GPU')
#tf.config.get_visible_devices('GPU')

# Fashion-MNIST Data

In [3]:
# Load the Fashion MNIST dataset
(X1, y1), (X2, y2) = tf.keras.datasets.fashion_mnist.load_data()
X = np.concatenate((X1, X2), axis=0)
X = X / 255.0  # Normalize the pixel values to [0, 1]
Y = np.concatenate((y1, y2))
print("Fashion MNIST Images shape:", X.shape)
print("Fashion MNIST Labels shape:", Y.shape)

Fashion MNIST Images shape: (70000, 28, 28)
Fashion MNIST Labels shape: (70000,)


In [ ]:
# Contamination introducer
def corrupt_labels(y_train, prob=0.2, num_classes=10, seed=None):
    if seed is not None:
        np.random.seed(seed)

    y_corrupted = y_train.copy()
    n = len(y_train)
    # Decide which labels to corrupt
    corrupt_mask = np.random.rand(n) < prob
    # For each label to corrupt, choose a new label different from the original
    for i in np.where(corrupt_mask)[0]:
        original_label = y_train[i]
        # possible new labels excluding the original
        new_labels = list(range(num_classes))
        new_labels.remove(original_label)
        # randomly pick a new label
        y_corrupted[i] = np.random.choice(new_labels)

    return y_corrupted

In [ ]:
# SD-loss implementation
class SDIV(Loss):
    def __init__(self, beta, lam, trim_ratio): # beta and lambda are the SD tuning parameter. Set trim_ratio to 0 to disable trimming, as done in our paper.
        super().__init__()
        self.beta = float(beta)
        self.lam = float(lam)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        A = 1 + self.lam*(1 - self.beta)
        B = self.beta - self.lam*(1 - self.beta)
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (tf.reduce_sum(y_pred**(self.beta+1), axis=1))/A - ((1+self.beta)/(A*B))*(sel_probs**B)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

# Trimmed CCE loss implementation
class TSCCE(Loss):
    def __init__(self, trim_ratio=0.2):
        super().__init__()
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        log_probs = tf.math.log(y_pred)
        # Get log probability of the correct class for each sample
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        per_sample_loss = -tf.gather_nd(log_probs, indices)
        # Trim top X% highest-loss samples
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_values, _ = tf.math.top_k(-per_sample_loss, k=k, sorted=False)
        trimmed_loss = -tf.reduce_mean(trimmed_values)

        return trimmed_loss

# DPD loss implementation
class TDPDSCCE(Loss):
    def __init__(self, beta, trim_ratio): # beta is the DPD tuning parameter. set trim_ratio to 0 to disable trimming, as done in our paper
        super().__init__()
        self.beta = float(beta)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  tf.reduce_sum(y_pred**(self.beta+1), axis=1) - (1+1/self.beta)*(sel_probs**self.beta)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

# SCE loss implementation
class SCE(Loss):
    def __init__(self, alpha, beta):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = float(beta)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0) # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses = -self.alpha*tf.math.log(sel_probs) + self.beta*6*(1 - sel_probs)

        return tf.reduce_mean(losses)

# GCE loss implementation
class GCE(Loss):
    def __init__(self, q):
        super().__init__()
        self.q = float(q)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (1 - (sel_probs**self.q))/self.q

        return tf.reduce_mean(losses)

# RKLD loss implementation
class RKLD(Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        
        losses =  tf.reduce_sum(y_pred*tf.math.log(y_pred), axis=1) + 2*(1 - sel_probs)        
        return tf.reduce_mean(losses)

# FCL loss implementation
class FCL(Loss):
    def __init__(self, mu):
        super().__init__()
        self.mu = float(mu)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32) # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =   (-tf.math.log(sel_probs))**(1-self.mu)/tf.exp(tf.math.lgamma(tf.constant(2-self.mu))) + 2*(1-sel_probs)

        return tf.reduce_mean(losses)

# Define NN model architecture

In [ ]:
def get_model():
    model = keras.Sequential([
        keras.layers.Flatten(),
        keras.layers.Dense(200, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(100, activation='relu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(10, activation='softmax', kernel_initializer=GlorotUniform(seed=42))
    ])
    return model

# 7-fold CV

In [ ]:
kf = KFold(n_splits=7, shuffle=True, random_state=42)

# rSDNet $(\beta, \lambda)$

In [ ]:
def ACC_SDIV(delta, beta, lam, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=SDIV(beta=beta, lam=lam, trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
betas = np.array([0.1,0.3,0.5,0.7,1.0]) # Choose your beta values for SD-loss here.
nj = len(dl); l_al = len(betas)
def inner_acc_sdiv(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    lam = -0.7 # Choose your value for lambda here.
    results = []
    for beta in betas:
        row_result = Parallel(n_jobs=nj)(delayed(ACC_SDIV)(delta, beta, lam, trim_ratio, X_train, y_train, X_test, y_test) for delta in dl)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2*nj)


def ACC_SDIV_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sdiv(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_SDIV_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
Sdiv_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = betas)
# Sdiv_cv.to_csv('SDIV.csv')
Sdiv_cv

# CCE

In [ ]:
def ACC_CCE(delta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
nj = len(dl)
def inner_acc_cce(X_train, y_train, X_test, y_test):
    row_result = Parallel(n_jobs=nj)(delayed(ACC_CCE)(delta, X_train, y_train, X_test, y_test) for delta in dl)
    return np.array(row_result).reshape(1,2*nj)

def ACC_CCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_cce(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=4)(delayed(ACC_CCE_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
df = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = ['CCE'])
# df.to_csv('CCE-CV.csv')
df

# TSCCE

In [ ]:
def ACC_TSCCE(delta, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=TSCCE(trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
t_ratios = [0.1,0.2,0.3] # these are the trimming ratios we consider for TSCCE in our experiments. You can change them as you like.
nj = len(dl); l_tr = len(t_ratios)
def inner_acc_tscce(X_train, y_train, X_test, y_test):
    results = []
    for tr in t_ratios:
        row_result = Parallel(n_jobs=nj)(delayed(ACC_TSCCE)(delta, tr, X_train, y_train, X_test, y_test) for delta in dl)
        results.append(row_result)
    return np.array(results).reshape(l_tr, 2*nj)

def ACC_TSCCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_tscce(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_TSCCE_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
tscce_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = t_ratios)
# tscce_cv.to_csv('TSCCE-CV.csv')
tscce_cv

# GCE

In [ ]:
def ACC_GCE(delta, q, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=GCE(q=q))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
qs = [0.5, 0.7] # these are the q values we consider for GCE in our experiments. You can change them as you like.
nj = len(dl); l_q = len(qs)
def inner_acc_gce(X_train, y_train, X_test, y_test):
    results = []
    for q in qs:
        row_result = Parallel(n_jobs=nj)(delayed(ACC_GCE)(delta, q, X_train, y_train, X_test, y_test) for delta in dl)
        results.append(row_result)
    return np.array(results).reshape(l_q, 2*nj)

def ACC_GCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_gce(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_GCE_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
gce_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = qs)
# gce_cv.to_csv('GCE-CV.csv')
gce_cv

# SCE

In [ ]:
def ACC_SCE(delta, alpha, beta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=SCE(alpha=alpha, beta=beta))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
alpha, beta = 0.5, 1.0 # these are the alpha and beta values we consider for SCE in our experiments. You can change them as you like.
nj = len(dl)
def inner_acc_sce(X_train, y_train, X_test, y_test):
    results = []
    row_result = Parallel(n_jobs=nj)(delayed(ACC_SCE)(delta, alpha, beta, X_train, y_train, X_test, y_test) for delta in dl)
    results.append(row_result)
    return np.array(results).reshape(1, 2*nj)

def ACC_SCE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_sce(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_SCE_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
sce_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = [f'alpha={alpha}, beta={beta}'])
# sce_cv.to_csv('SCE-CV.csv')
sce_cv

# FCL

In [ ]:
def ACC_FCL(delta, mu, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=FCL(mu=mu))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
mus = [0.0,0.25,0.5,0.75] # these are the mu values we consider for FCL in our experiments. You can change them as you like.
nj = len(dl); l_mu = len(mus)
def inner_acc_fcl(X_train, y_train, X_test, y_test):
    results = []
    for mu in mus:
        row_result = Parallel(n_jobs=nj)(delayed(ACC_FCL)(delta, mu, X_train, y_train, X_test, y_test) for delta in dl)
        results.append(row_result)
    return np.array(results).reshape(l_mu, 2*nj)

def ACC_FCL_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_fcl(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_FCL_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
fcl_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = mus)
# fcl_cv.to_csv('FCL-MNIST-CV.csv')
fcl_cv

# TDPDSCCE

In [ ]:
def ACC_TDPD(delta, beta, trim_ratio, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=TDPDSCCE(beta=beta, trim_ratio=trim_ratio))
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
al = np.array([0.1,0.3,0.5,0.7,1]) # these are the beta values we consider for TDPD in our experiments. You can change them as you like.

nj = len(dl); l_al = len(al)
def inner_acc_tdpd(X_train, y_train, X_test, y_test):
    trim_ratio = 0.0
    results = []
    for beta in al:
        row_result = Parallel(n_jobs=nj)(delayed(ACC_TDPD)(delta, beta, trim_ratio, X_train, y_train, X_test, y_test) for delta in dl)
        results.append(row_result)
    return np.array(results).reshape(l_al, 2*nj)

def ACC_TDPD_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_tdpd(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_TDPD_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
tdpd_cv = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = al)
# tdpd_cv.to_csv('TDPD-CV.csv')
tdpd_cv

# MAE

In [ ]:
from keras.utils import to_categorical

def ACC_MAE(delta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    y_train_corrupted = to_categorical(y_train_corrupted, 10)
    model.compile(optimizer='adam', loss='mae')
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
nj = len(dl)
def inner_acc_mae(X_train, y_train, X_test, y_test):
    row_result = Parallel(n_jobs=nj)(delayed(ACC_MAE)(delta, X_train, y_train, X_test, y_test) for delta in dl)
    return np.array(row_result).reshape(1,2*nj)

def ACC_MAE_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_mae(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_MAE_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
df = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = ['MAE'])
# df.to_csv('MAE-CV.csv')
df

# RKLD

In [ ]:
def ACC_RKLD(delta, X_train, y_train, X_test, y_test):
    y_train_corrupted = corrupt_labels(y_train, prob=delta, num_classes=10, seed=42)
    y_test_corrupted = corrupt_labels(y_test, prob=delta, num_classes=10, seed=42)
    np.random.seed(42)
    tf.random.set_seed(42)
    model = get_model()
    model.compile(optimizer='adam', loss=RKLD())
    model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
    y_pred = model.predict(X_test, verbose=0)
    y_pred_labels = [np.argmax(i) for i in y_pred]
    val, val_cr = accuracy_score(y_test, y_pred_labels), accuracy_score(y_test_corrupted, y_pred_labels)
    return val, val_cr

dl = [0, 0.1, 0.2, 0.3, 0.4, 0.5] # these are the label corruption ratios we consider in our experiments. You can change them as you like.
nj = len(dl)
def inner_acc_rkld(X_train, y_train, X_test, y_test):
    row_result = Parallel(n_jobs=nj)(delayed(ACC_RKLD)(delta, X_train, y_train, X_test, y_test) for delta in dl)
    return np.array(row_result).reshape(1,2*nj)

def ACC_RKLD_fold(train_idx, val_idx):
    X_train, X_test = X[train_idx], X[val_idx]  # Convert to NumPy array
    y_train, y_test = Y[train_idx], Y[val_idx]  # Convert to NumPy array
    return inner_acc_rkld(X_train, y_train, X_test, y_test)

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=7)(delayed(ACC_RKLD_fold)(train_idx, val_idx) for train_idx, val_idx in kf.split(Y))
df = pd.DataFrame(np.mean(results, axis=0), columns = np.repeat(dl,2), index = ['RKLD'])
# df.to_csv('RKLD-CV-new.csv')
df